In [ ]:
# ======================================================
# Notebook: 2D contamination detection
# Bayesian-style sampling + hyperparameter tuning
# ======================================================

import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import RandomizedSearchCV
from sklearn.svm import SVC

# Load data
X = np.load("/mnt/data/initial_inputs.npy")      # (N,2)
y = np.load("/mnt/data/initial_outputs.npy")     # (N,)
y_bin = (y > 0).astype(int)

# Hyperparameter tuning (SVM)
param_dist = {
    "C": np.logspace(-2, 3, 20),
    "gamma": np.logspace(-3, 2, 20),
    "kernel": ["rbf"]
}

search = RandomizedSearchCV(
    SVC(probability=True),
    param_distributions=param_dist,
    n_iter=25,
    cv=3,
    random_state=42
)

search.fit(X, y_bin)
model = search.best_estimator_

print("Best params:", search.best_params_)

# Candidate grid
x_min, x_max = X[:,0].min(), X[:,0].max()
y_min, y_max = X[:,1].min(), X[:,1].max()

gx = np.linspace(x_min, x_max, 100)
gy = np.linspace(y_min, y_max, 100)
XX, YY = np.meshgrid(gx, gy)
X_grid = np.vstack([XX.ravel(), YY.ravel()]).T

# Predict probabilities
probs = model.predict_proba(X_grid)[:,1]

# Acquisition: uncertainty (decision boundary)
acquisition = -np.abs(probs - 0.5)

# Select next (10,2) inputs
top_idx = np.argsort(acquisition)[-10:]
next_points = X_grid[top_idx]

print("Next (10,2) inputs:")
print(next_points)

# Plot
plt.figure(figsize=(6,5))
plt.contourf(XX, YY, probs.reshape(XX.shape), levels=30)
plt.scatter(X[:,0], X[:,1])
plt.scatter(next_points[:,0], next_points[:,1], marker='x', s=100)
plt.show()